# Chapter 12 — Data and loops

Step 11, and the language grows a body: statement `if`/`for`, arrays as
captures, a **C backend** (the seam's first non-GPU target), and — a
user-directed exercise — **named axes**, the xarray idea, done pedantically.
Design: `docs/design/100_arrays-and-axes.md`; the punning charter it
implements is `090_core-and-extensions.md`.

The kernel was finished two chapters ago, and it shows: everything in this
chapter is satellite registrations — one kernel line was spent (the `device`
field on `Array`, 090's dispatch axis, waiting for step 14), and nothing
else moved.

In [1]:
import numpy as np

import pdum.dsl  # noqa: F401
from pdum.dsl import jit, no_compile
from pdum.dsl.kernel.registry import DEFAULT

## Statements: strict joins, bounded loops

`if`/`else` statements lower to `core.if` yielding the JOIN of rebound
locals — and the join is **strict**: a name assigned on either path must
have the SAME type on both, no unification, no fixpoint (ch07a's "what type
inference is here" promise, kept). A name born in only ONE suite is a
branch-local temporary: it dies with its suite (reading it later is a loud
name error), exactly as loop-locals die with their loop. `for i in range(...)` lowers to
`core.for` with the rebound pre-existing locals as **loop carries**. The
loop variable dies with the loop; `while`, `break`, `continue` are refused
by POLICY — bounded loops are the GPU-honest subset every serious kernel
language settles on (R11).

In [2]:
def make(gain):
    @jit()
    def f(x):
        total = 0.0
        count = 0
        for i in range(10):
            v = float(i) * gain
            if v > 2.0:
                total = total + v
                count = count + 1
            else:
                total = total + x
        return total + float(count)

    return f


f = make(0.5)
print("f(1.5) =", f(1.5))
c0 = DEFAULT.specializations.compiles
with no_compile():  # a fresh closure, same types: the thesis, now with control flow
    make(0.7)(2.5)
print("fresh closure -> compiles:", DEFAULT.specializations.compiles - c0)

f(1.5) = 30.0
fresh closure -> compiles: 0


In [3]:
key = DEFAULT.specializations.key_for(f, (DEFAULT.table.fingerprint(1.5),), DEFAULT.backend_for(f.kind).fp)
print(DEFAULT.specializations.probe(key).artifact.__pdum_source__)

import math
from struct import unpack_from as _u
from pdum.dsl.kernel.registry import Out as _Out

def _tdiv(a, b):  # exact trunc division (numeric policy: C semantics)
    q = a // b
    return q + 1 if q < 0 and q * b != a else q

def _tmod(a, b):
    return a - _tdiv(a, b) * b

def kernel(staging, leaves):
    if any(isinstance(x, _Out) for x in leaves):
        raise TypeError("the python backend takes no launcher data (out= is for device targets)")
    v0 = 0
    v1 = 10
    v2 = 0.0
    v3 = 0
    v4 = (v2, v3)
    v25 = v4
    for v5 in range(v0, v1):
        v11 = v25
        v6 = float(v5)
        v7 = _u('<d', staging, 0)[0]
        v8 = v6 * v7
        v9 = 2.0
        v10 = v8 > v9
        v12 = v11[0]
        v14 = v11[1]
        if v10:
            v13 = v12 + v8
            v15 = 1
            v16 = v14 + v15
            v17 = (v13, v16)
            v21 = v17
        else:
            v18 = _u('<d', staging, 8)[0]
            v19 = v12 + v18
            v20 = (v19, v14)

Read the render: the loop carries `(total, count)` as ONE tuple value, the
branch is a real lazy `if`/`else` (dominator placement — work shared by both
paths hoists, branch-exclusive work stays inside), and `v10 > 2.0` gates a
staging read that only the else-path needs. Guard-then-divide keeps working
inside loops for the same reason it worked in ch09.

In [4]:
# The refusals ARE the language definition — each names its policy:
@jit()
def wants_while(x):
    while x > 0.0:
        x = x - 1.0
    return x


@jit()
def wants_break(x):
    acc = 0.0
    for i in range(3):
        acc = acc + x
        break
    return acc


@jit()
def early_return(x):
    if x > 0.0:
        return x
    return -x


@jit()
def one_sided(x):
    if x > 0.0:
        y = x  # noqa: F841 — the refusal below IS the lesson
    else:
        pass
    return x


@jit()
def bad_join(x):
    y = 0.0
    if x > 0.0:
        y = 1
    else:
        y = 1.0
    return float(y)


for k in (wants_while, wants_break, early_return, one_sided, bad_join):
    try:
        k(1.0)
        print(f"  ✅ {k.fntype.template.label}")
    except Exception as e:
        print(f"  🚫 {type(e).__name__}: {str(e)[:100]}")

  🚫 MissingRule: `while` is not in the base pack: bounded `for i in range(...)` only — every serious kernel language 
  🚫 MissingRule: `break` is not in the base pack: loops are single-entry single-exit; guard with `if` instead [/var/f
  🚫 MissingRule: single tail return: `return` must be the function body's last statement, not inside a branch or loop
  🚫 MissingRule: this `if` binds nothing that survives it — dead code is refused [/var/folders/vw/tdwty7n902d81s2lj6n
  🚫 TypeError: strict join: 'y' is i64 on the then-path but f64 on the else-path [/var/folders/vw/tdwty7n902d81s2lj


## Arrays are captures — and shape is a VALUE

A numpy array captured by a kernel is summarized by `typeof` like every
capture: `Array(dtype, rank, layout, device)` — **rank, not shape**. Shape
and strides ride the staging buffer as i64 slots (uniforms, in GPU terms);
the payload travels the leaves channel as a pointer. So the thesis extends
to data: *a new shape is a cache HIT* — only rank, dtype, or device changes
recompile.

The classic use: a color table. Map a scalar through a captured palette —
three loads and a luminance dot:

In [5]:
palette = np.array(
    [[0.267, 0.005, 0.329], [0.229, 0.322, 0.546], [0.128, 0.567, 0.551], [0.993, 0.906, 0.144]]
)  # viridis, heavily abridged


def colorize(table):
    n = table.shape[0] - 1  # host-side int: a plain i64 capture

    @jit()
    def lum(u):
        k = int(u * float(n) + 0.5)
        return 0.2126 * table[k, 0] + 0.7152 * table[k, 1] + 0.0722 * table[k, 2]

    return lum


lum = colorize(palette)
print("lum(0.0) =", lum(0.0), " lum(1.0) =", lum(1.0))
c0 = DEFAULT.specializations.compiles
with no_compile():  # an 8-row palette: DIFFERENT shape, same rank -> hit
    colorize(np.ones((8, 3)))(0.5)
print("new shape, same rank -> compiles:", DEFAULT.specializations.compiles - c0)
palette[3] = 0.0  # in-place edit: buffers are DATA (the pointer travels per call)
print("after palette edit, lum(1.0) =", lum(1.0))

lum(0.0) = 0.08409400000000002  lum(1.0) = 0.8694798
new shape, same rank -> compiles: 0
after palette edit, lum(1.0) = 0.0


In [6]:
key = DEFAULT.specializations.key_for(lum, (DEFAULT.table.fingerprint(0.0),), DEFAULT.backend_for(lum.kind).fp)
print(DEFAULT.specializations.probe(key).artifact.__pdum_source__)

import math
from struct import unpack_from as _u
from pdum.dsl.kernel.registry import Out as _Out

def _tdiv(a, b):  # exact trunc division (numeric policy: C semantics)
    q = a // b
    return q + 1 if q < 0 and q * b != a else q

def _tmod(a, b):
    return a - _tdiv(a, b) * b

def kernel(staging, leaves):
    if any(isinstance(x, _Out) for x in leaves):
        raise TypeError("the python backend takes no launcher data (out= is for device targets)")
    v0 = 0.2126
    v1 = leaves[0]
    v2 = _u('<d', staging, 40)[0]
    v3 = _u('<q', staging, 0)[0]
    v4 = float(v3)
    v5 = v2 * v4
    v6 = 0.5
    v7 = v5 + v6
    v8 = int(v7)
    v9 = _u('<q', staging, 24)[0]
    v10 = v8 * v9
    v11 = 0
    v12 = _u('<q', staging, 32)[0]
    v13 = v11 * v12
    v14 = v10 + v13
    v15 = v1[v14]
    v16 = v0 * v15
    v17 = 0.7152
    v18 = leaves[0]
    v19 = _u('<q', staging, 24)[0]
    v20 = v8 * v19
    v21 = 1
    v22 = _u('<q', staging, 32)[0]
    v23 = v21 * v22
    v24 = v20 + v23
  

`leaves[0]` is the payload pointer; the `_u('<q', staging, ...)` reads are
the STRIDES, packed per call like any uniform. That is what "rank-generic"
costs at runtime: two multiplies. And it is what the hit across shapes buys.

## The dial: shape in the type when you want it

§13 calls `typeof` a *summary function* with a dial. `Shaped(x)` turns it
one notch: the full shape enters the type — one specialization per shape,
strides become compile-time CONSTANTS, and the shape/stride slots leave
staging entirely:

In [7]:
from pdum.dsl.stdlib.arrays import Shaped


def colorize_shaped(table):
    n = table.shape[0] - 1

    @jit()
    def lum(u):
        k = int(u * float(n) + 0.5)
        return 0.2126 * table[k, 0] + 0.7152 * table[k, 1] + 0.0722 * table[k, 2]

    return lum


s = colorize_shaped(Shaped(np.ascontiguousarray(palette)))
s(0.5)
c0 = DEFAULT.specializations.compiles
colorize_shaped(Shaped(np.ones((8, 3))))(0.5)  # new shape IS a new type now
print("Shaped, new shape -> compiles:", DEFAULT.specializations.compiles - c0)
fp = (DEFAULT.table.fingerprint(0.5),)
rec = DEFAULT.specializations.probe(DEFAULT.specializations.key_for(s, fp, DEFAULT.backend_for(s.kind).fp))
print("staging bytes with shape in the type:", rec.plan.staging_size, "(just the f64 argument)")

Shaped, new shape -> compiles: 1
staging bytes with shape in the type: 16 (just the f64 argument)


Neither notch is "right". Rank-generic is the render-loop default (palettes
get resized); shape-in-type is the numerics default (unroll against a fixed
64×64 tile). The dial is per-VALUE, the mechanism is one `typeof` choice,
and both notches were already paid for by the two-tier cache.

## Named axes — the xarray exercise

A user-directed detour (100 §2): xarray names its axes, and *most kernel
languages pretend that idea does not exist*. We take the opposite, pedantic
position. `Named(a, ("y", "x"))` — or any `xarray.DataArray` — summarizes as
`NamedArray(..., dims)`: the names live IN the type. The consequences
follow mechanically:

- `isel(y=…, x=…)` keywords are **mandatory**; keyword ORDER is free.
- Positional indexing on a named array is **refused** — transposition is
  precisely the bug names exist to kill, so there is no back door.
- Renaming axes is a *different type* (recompile — and stale `isel` calls
  refuse loudly at lower time, naming the axes that do exist).
- The names are GONE by codegen. Zero runtime cost — because names live on
  the types-not-values side of the line, like every identity in this
  system.

In [8]:
from pdum.dsl.stdlib.arrays import Named

field = Named(np.arange(12.0).reshape(3, 4), ("y", "x"))


def sample(t):
    @jit()
    def at(a, b):
        return t.isel(x=b, y=a)  # note the order: names bind, position is dead

    return at


at = sample(field)
print("isel(y=1, x=2) =", at(1, 2), "== numpy [1,2] =", field.array[1, 2])


def sample_positional(t):
    @jit()
    def at(a, b):
        return t[a, b]

    return at


try:
    sample_positional(field)(0, 0)
except Exception as e:
    print("🚫", str(e)[:110])

isel(y=1, x=2) = 6.0 == numpy [1,2] = 6.0
🚫 this array's axes have NAMES ('y', 'x') — positional indexing is refused; use .isel(y=..., x=...) [/var/folder


In [9]:
import xarray as xr

da = xr.DataArray(np.arange(12.0).reshape(3, 4), dims=("lat", "lon"))


def sample_xr(t):
    @jit()
    def at(a, b):
        return t.isel(lon=b, lat=a)

    return at


print("xarray isel:", sample_xr(da)(2, 3))
c0 = DEFAULT.specializations.compiles
with no_compile():  # a FRESH DataArray, same dims/dtype/rank: same type, hit
    sample_xr(xr.DataArray(np.zeros((9, 9)), dims=("lat", "lon")))(0, 0)
print("fresh DataArray -> compiles:", DEFAULT.specializations.compiles - c0)
try:
    sample_xr(xr.DataArray(np.ones((2, 2)), dims=("north", "east")))(0, 0)
except Exception as e:
    print("🚫 renamed dims:", str(e)[:90])

xarray isel: 11.0
fresh DataArray -> compiles: 0
🚫 renamed dims: isel is pedantic on purpose: name every axis exactly once as a keyword — expected {'north'


### The free lunch, proven by the artifact cache

The pedantry costs nothing — and the system can PROVE it. A positional
kernel over an anonymous array and an `isel` kernel over a named one lower
to IDENTICAL IR (same stride arithmetic, same env paths; the names never
reach a node). Identical IR means an identical content key, and tier 2 is
content-addressed:

In [10]:
ext = DEFAULT.extend()  # fresh caches, same vocabulary
raw = np.arange(6.0).reshape(2, 3)


def pos(t):
    @jit()
    def k(i, j):
        return t[i, j]

    return k


def named(t):
    @jit()
    def k(i, j):
        return t.isel(y=i, x=j)

    return k


fp2 = (ext.table.fingerprint(0),) * 2
ext.dispatch(pos(raw), (0, 1))
ext.dispatch(named(Named(raw, ("y", "x"))), (0, 1))
print("specializations:", len(ext.specializations._ready), " artifacts:", len(ext.artifacts))
print("two source spellings, two cache-key identities, ONE compiled artifact")

specializations: 2  artifacts: 1
two source spellings, two cache-key identities, ONE compiled artifact


## The C target: the seam beyond GPUs

`backends/c.py` is the contribution point's first *citizen* (080's
namespace package stops being infra-only): rendered C99, compiled by the
system `cc`, loaded with `ctypes` — a third source renderer over the same
IR and a second runtime shape (dlopen vs `exec` vs wgpu) under the ONE
calling convention `launch(staging, leaves)`. It never claims the default
route; choosing it is explicit (a child registry — 080's tier-2 override)
until the `device` axis brings data-driven dispatch at step 14.

In [11]:
from pdum.dsl.backends import c

print("cc on PATH:", c.is_available())
cext = DEFAULT.extend()
c.install(cext, default=True)

print("twin :", f(1.5))
print("C    :", cext.dispatch(f, (1.5,)))
print("lum twin/C:", lum(0.75), cext.dispatch(lum, (0.75,)))
print("named on C:", cext.dispatch(at, (1, 2)))

cc on PATH: True
twin : 30.0


C    :

 30.0
lum twin/C: 0.4725133999999999 0.4725133999999999


named on C: 6.0


In [12]:
kc = cext.specializations.key_for(f, (cext.table.fingerprint(1.5),), cext.backend_for(f.kind).fp)
print(cext.specializations.probe(kc).artifact.__pdum_source__)

/* pdum-restype: f64 */
#include <stdint.h>
#include <stdbool.h>
#include <string.h>
#include <math.h>
#define LD(name, T, n) static inline T name(const unsigned char* p){ T v; memcpy(&v, p, n); return v; }
LD(ld_f64, double, 8) LD(ld_f32, float, 4) LD(ld_i64, int64_t, 8) LD(ld_i32, int32_t, 4)
LD(ld_u64, uint64_t, 8) LD(ld_u32, uint32_t, 4) LD(ld_b, bool, 1)

#if defined(_WIN32)
__declspec(dllexport)
#endif
double kernel(const unsigned char* staging, void* const* bufs) {
  int64_t v0 = 0LL;
  int64_t v1 = 10LL;
  double v2 = 0.0;
  int64_t v3 = 0LL;
  double v4_0 = v2; int64_t v4_1 = v3;
  double v25_0; int64_t v25_1;
  v25_0 = v4_0;
  v25_1 = v4_1;
  for (int64_t v5 = v0; v5 < v1; ++v5) {
    double v11_0; int64_t v11_1;
    v11_0 = v25_0;
    v11_1 = v25_1;
    double v6 = (double)(v5);
    double v7 = ld_f64(staging + 0);
    double v8 = v6 * v7;
    double v9 = 2.0;
    bool v10 = v8 > v9;
    double v12 = v11_0;
    int64_t v14 = v11_1;
    double v21_0; int64_t v21_1;
    if (v1

Things to read in the C: the loop carry is SCALARIZED (`v25_0, v25_1` —
tuples become lane variables, no structs), staging reads are `memcpy`
inlines the compiler folds to plain loads, and the buffer parameter is
`bufs[k]` — the leaves channel, exactly as the plan ordered it.

## The ray-march spike

The step's exit question (020): can this language express a real iterative
kernel — a sphere tracer: 32 bounded steps, a branch, a multi-carry, an
intrinsic — and what do the instruments (ch11b) say about running it
per-pixel on the CPU?

In [13]:
from pdum.dsl.bench import benchmark


def make_marcher(cx, cy, cz, radius):
    @jit()
    def march(ox, oy):
        t = 0.0
        hit = 0.0
        for i in range(32):
            dx = ox - cx
            dy = oy - cy
            dz = -3.0 + t - cz
            d = sqrt(dx * dx + dy * dy + dz * dz) - radius  # noqa: F821
            if d < 0.001:
                hit = 1.0
            t = t + max(d, 0.001)  # noqa: F821
        return hit * (1.0 - t / 6.0)

    return march


m = make_marcher(0.0, 0.0, 0.5, 1.0)
print("center ray:", m(0.0, 0.0), " edge ray:", m(2.0, 2.0))
print("C agrees:", abs(cext.dispatch(m, (0.0, 0.0)) - m(0.0, 0.0)) < 1e-9)

t_py = benchmark(lambda: m(0.1, 0.2), budget_s=0.25)
t_c = benchmark(lambda: cext.dispatch(m, (0.1, 0.2)), budget_s=0.25)
print("python twin:", t_py)
print("C target   :", t_c)
u = 1e6
print(
    f"body speedup ~{t_py.minimum / t_c.minimum:.1f}x; a 64x64 frame at per-ray dispatch: "
    f"~{t_c.minimum * 4096 * 1e3:.1f} ms on C"
)

center ray: 0.5781666666666672  edge ray: -0.0


C agrees:

 True


python twin: Trial(min 22.46 µs · median 23.50 µs · mean 25.69 µs · 9731 samples × 1 evals)
C target   : Trial(min 3.49 µs · median 3.59 µs · mean 3.68 µs · 4241 samples × 16 evals)
body speedup ~6.4x; a 64x64 frame at per-ray dispatch: ~14.3 ms on C


## The verdict

**Expressiveness: GO.** A sphere tracer is `for` + `if` + carries +
batteries; the render reads like the shader you would have written.

**Granularity: the honest finding.** On the M3 the C body runs a 32-step
march in well under a microsecond — but ~2.4 µs of every per-ray call is
DISPATCH (ch11b's decomposition: key+probe, extract, pack). Per-pixel ×
per-call is simply the wrong shape for CPU frames: the C target wins 7× on
the body and then drowns in per-call overhead that the GPU path amortizes
over the whole domain (ch10's `out=(W,H)` launches 65k threads on ONE
dispatch). CPU frames want the same move — a domain call or DPS
out-arrays — and that is chaining/step-14 territory (100 §6), not a reason
to contort v1.

## Things to notice

- **One kernel line** this whole step (`Array.device`, the 090 axis).
  Statements, arrays, named axes, and a C backend: all satellites. The
  extension-locality law held through the widest step since the kernel
  froze.
- **Loop binders carry TUPLE indices** — `("loop", *inline-prefix, seq)`.
  `core.param` identity is structural, so reusing integer index 0 for a
  loop variable would make two different programs hash identically (an
  artifact-cache collision); and a shared counter would make content keys
  depend on process HISTORY (review-caught). The tuple is unique across
  inlining and deterministic from source order alone — cache keys cannot
  drift between runs.
- **The pedantic pick is free.** Named axes cost one wrapper (or arrive
  free from xarray), are enforced at lower time, and compile to the SAME
  artifact as positional code. Pedantry priced at zero is the two-tier
  cache doing exactly what ch03 promised.
- Next: transforms — vmap and the jvp precursor (step 12), where these
  loops and arrays meet their first IR-to-IR columns.